# Create round_info.csv

Builds `round_info.csv` from the HAL configs (notebook 01) and the FOV/boundary layout (notebook 02) -- the per-round series/HAL-config/data-dir table that notebook 04 turns into the Dave recipe.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.dave      import (
    create_round_info, create_round_info_multitissue, create_data_drive_skeleton,
)
from MERci.acquisition.positions import discover_boundary_files

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAME = SAMPLE_DIR.name
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────
MICROSCOPE  = "ST2"   # microscope identifier
DATA_DRIVES = ["C:", "G:"] #["E", "G:", "F:"]      # e.g. ["D:", "E:", "F:"] to round-robin hyb rounds across physical
                                # drives (cells/transit stay on SAMPLE_DIR's own drive); [] = single-drive

# ── Detect the tissue/boundary layout (written by notebook 02) ──────
# >1 boundary -> per-segment recipe (boundary + transit movies); else the classic
# single-positions recipe.
boundaries, MODE = discover_boundary_files(POSITIONS_DIR)
MULTI_BOUNDARY   = len(boundaries) > 1
print(f"Layout mode: {MODE}  ({len(boundaries)} boundary file(s)) -> "
      f"{'per-segment' if MULTI_BOUNDARY else 'single-positions'} recipe")

# HAL config filenames (from notebook 01 / SETTINGS_DIR)
# Adjust these to match the actual files created by notebook 01
bits_hal_configs    = sorted(SETTINGS_DIR.glob("hal-config-*bits*.xml"))
cells_hal_configs   = sorted(SETTINGS_DIR.glob("hal-config-*cells*.xml"))
transit_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*transit*.xml"))

print("\nAvailable HAL configs in settings/:")
for p in sorted(SETTINGS_DIR.glob("hal-config-*.xml")):
    print(f"  {p.name}")

# Set these manually if auto-detection picks the wrong files
BITS_HAL_CONFIG    = bits_hal_configs[0].name    if bits_hal_configs    else "hal-config-mf3-bits.xml"
CELLS_HAL_CONFIG   = cells_hal_configs[0].name   if cells_hal_configs   else "hal-config-mf3-cells.xml"
TRANSIT_HAL_CONFIG = transit_hal_configs[0].name if transit_hal_configs else None

print(f"\nBits    HAL config : {BITS_HAL_CONFIG}")
print(f"Cells   HAL config : {CELLS_HAL_CONFIG}")
print(f"Transit HAL config : {TRANSIT_HAL_CONFIG}")

if MULTI_BOUNDARY and TRANSIT_HAL_CONFIG is None:
    raise FileNotFoundError(
        "Multiple boundaries detected but no hal-config-*transit*.xml in settings/. "
        "Run the transit cell in notebook 01 first."
    )

## Round – bit – color mapping

Define the round → bit → colour mapping for the codebook. This is the single
source of **`N_HYBS`** (the number of hybridisation/bits rounds, taken as the max
round index) used by the recipe below, and it is saved to `round_bit_color_map.csv`
for notebook 05 to reuse (data organization + Dave annotation).

In [ ]:
# round : hyb/bit index (1-indexed), matching the bits movie series number
#         (hal-{mic}_01, _02, …); NOT the Dave imaging-round number.
# bit   : bit number     |     color : excitation wavelength (nm)
round_bit_color = [
    (1,  1,  647), (1,  2,  560),
    (2,  3,  560), (2,  4,  647),
    (3,  5,  647), (3,  6,  560),
    (4,  7,  647), (4,  8,  560),
    (5,  9,  560), (5,  10, 647),
    (6,  11, 647), (6,  12, 560),
    (7,  13, 647), (7,  14, 560),
    (8,  15, 560), (8,  16, 647),
    (9,  17, 647), (9,  18, 560),
    (10, 19, 647), (10, 20, 560),
    (11, 21, 560), (11, 22, 647),
    (12, 23, 647), (12, 24, 560),
    (13, 25, 647), (13, 26, 560),
]

rbc_df   = pd.DataFrame(round_bit_color, columns=["round", "bit", "color"])
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
rbc_df.to_csv(rbc_path, index=False)

N_HYBS = int(rbc_df["round"].max())   # number of bits rounds, derived from the mapping
print(f"Saved: {rbc_path}")
print(f"N_HYBS (from mapping): {N_HYBS}")
print(rbc_df.to_string(index=False))

In [ ]:
if DATA_DRIVES:
    create_data_drive_skeleton(
        sample_dir  = SAMPLE_DIR,
        n_bits      = N_HYBS,
        data_drives = DATA_DRIVES,
        mode        = MODE,
        boundaries  = boundaries if MODE == "multi" else None,
    )

if MULTI_BOUNDARY:
    round_info = create_round_info_multitissue(
        microscope         = MICROSCOPE,
        n_bits             = N_HYBS,
        bits_hal_config    = BITS_HAL_CONFIG,
        cells_hal_config   = CELLS_HAL_CONFIG,
        transit_hal_config = TRANSIT_HAL_CONFIG,
        sample_dir         = SAMPLE_DIR,
        boundaries         = boundaries,
        mode               = MODE,
        sample_name        = SAMPLE_NAME,
        data_drives        = DATA_DRIVES or None,
    )
else:
    round_info = create_round_info(
        microscope       = MICROSCOPE,
        n_bits           = N_HYBS,
        bits_hal_config  = BITS_HAL_CONFIG,
        cells_hal_config = CELLS_HAL_CONFIG,
        sample_dir       = SAMPLE_DIR,
        data_drives      = DATA_DRIVES or None,
    )

print(round_info.to_string(index=False))

out_csv = METADATA_DIR / "round_info.csv"
round_info.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")